# Dana's Bakery — deterministic LangGraph
Run all cells in Google Colab. No API keys or LLM calls are used.

Each order uses an unchanged inventory snapshot (no stock depletion between
invocations). Empty city means pickup; another city triggers a friendly pickup
conversion. Substitution is automatic: each next candidate counts as one attempt,
including unavailable candidates, and at most two candidates are checked.
On rejection, `item` records the last attempted product and `total` is zero.
Prices use Decimal arithmetic; the output total is a float in shekels.
Accepted messages include itemized receipts, discount explanations, and any
original-to-final product substitution, keeping the same six output keys.

The diagram cell uses `draw_mermaid_png()`, whose default renderer requires
network access to Mermaid's rendering service. Order processing runs locally.


In [ ]:
%pip install -q "langgraph>=1.0,<2.0"

In [ ]:
"""Deterministic Dana's Bakery workflow. Install: pip install -r requirements.txt."""
from decimal import Decimal
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

PRICES = {'challah': 18, 'babka': 45, 'rugelach': 30, 'sourdough': 25}
INITIAL_STOCK = {'challah': 10, 'babka': 0, 'rugelach': 3, 'sourdough': 4}
FALLBACK = {'babka': 'rugelach', 'rugelach': 'sourdough', 'sourdough': 'challah'}


class OrderInput(TypedDict):
    item: str
    quantity: int
    day: str
    city: str


class OrderOutput(TypedDict):
    status: str
    item: str
    delivery: str
    total: float
    message: str
    steps: list[str]


class State(OrderInput, OrderOutput):
    suggestions_count: int
    original_item: str


def build_graph(stock=None):
    """Each invocation checks an unchanged stock snapshot; orders are quotations.

    No stock is consumed between invocations. Pass a stock mapping to model a
    different inventory snapshot. Unavailable candidates count as suggestions;
    the first candidate with enough stock is automatically selected.
    """
    inventory = dict(INITIAL_STOCK if stock is None else stock)
    if set(inventory) != set(PRICES) or any(type(n) is not int or n < 0 for n in inventory.values()):
        raise ValueError('Stock must contain all four active products with nonnegative integer counts.')

    def initialize(s):
        return {'item': s.get('item', ''), 'quantity': s.get('quantity', 0),
                'day': s.get('day', ''), 'city': s.get('city', ''),
                'status': 'accepted', 'delivery': 'pickup', 'total': 0.0,
                'message': '', 'steps': ['initialize: order received'], 'suggestions_count': 0,
                'original_item': ''}

    def validate(s):
        item = s['item'].strip().lower() if isinstance(s['item'], str) else ''
        reason = ''
        if item not in PRICES:
            reason = f"I'm sorry, {item or 'that item'} isn't on our active menu."
        elif type(s['quantity']) is not int or s['quantity'] <= 0:
            reason = "I'm sorry, please order a positive whole-number quantity."
        elif not isinstance(s['day'], str) or s['day'].strip().lower() not in {
            'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'
        } or not isinstance(s['city'], str):
            reason = "I'm sorry, please provide a valid weekday and a city as text."
        return {'item': item, 'original_item': item, 'status': 'rejected' if reason else 'accepted',
                'message': reason,
                'steps': s['steps'] + ['validate: ' + (reason or 'valid order')]}

    def delivery(s):
        city = s['city'].strip()
        mode = 'delivery' if city.casefold() == 'jerusalem' else 'pickup'
        note = ("I'm sorry, we only deliver inside Jerusalem, so I've arranged free pickup for you. "
                if city and mode == 'pickup' else '')
        return {'delivery': mode, 'message': note,
                'steps': s['steps'] + [f'delivery: {mode}; fee {20 if mode == "delivery" else 0}₪']}

    def check_stock(s):
        enough = inventory[s['item']] >= s['quantity']
        return {'steps': s['steps'] + [f'stock: {s["item"]}; ' + ('sufficient' if enough else 'insufficient')]}

    def route_stock(s):
        if inventory[s['item']] >= s['quantity']:
            return 'price'
        return 'substitute' if s['suggestions_count'] < 2 and s['item'] in FALLBACK else 'reject'

    def substitute(s):
        candidate = FALLBACK[s['item']]
        count = s['suggestions_count'] + 1
        return {'item': candidate, 'suggestions_count': count,
                'message': s['message'] + f"We're short of {s['item']}; let me try {candidate} for you. ",
                'steps': s['steps'] + [f'substitute: suggestion {count}: {candidate}']}

    def reject(s):
        return {'status': 'rejected', 'total': 0.0,
                'message': s['message'] + "I'm so sorry, we can't fill your full quantity today, so your order is cancelled. We'd love to bake for you another day! — Dana",
                'steps': s['steps'] + ['reject: no fulfillable substitute within two suggestions']}

    def price(s):
        goods = Decimal(PRICES[s['item']]) * s['quantity']
        friday = s['day'].strip().lower() == 'friday'
        discount = goods * Decimal('0.15') if goods > 150 and not friday else Decimal(0)
        fee = 20 if s['delivery'] == 'delivery' else 0
        total = (goods - discount + fee).quantize(Decimal('0.01'))
        receipt = [f"Lovely! Your {s['quantity']} × {s['item']} are reserved for {s['delivery']}."]
        if s['suggestions_count']:
            receipt.append(f"Substitution: {s['original_item']} → {s['item']} (same quantity: {s['quantity']}).")
        receipt.extend(['', f"Baked goods: {s['quantity']} × {PRICES[s['item']]:.2f}₪ = {goods:.2f}₪"])
        if discount:
            receipt.append(f'Your 15% baked-goods discount: −{discount:.2f}₪')
        elif friday:
            receipt.append('Discount: 0.00₪ — discounts take a break on Fridays.')
        else:
            receipt.append('Discount: 0.00₪ — 15% off applies when baked goods exceed 150.00₪.')
        receipt.extend([f'Jerusalem delivery: {fee:.2f}₪' if fee else 'Pickup: 0.00₪',
                        f'Total: {total:.2f}₪', '', 'Thank you for ordering! — Dana'])
        prefix = s['message'].strip()
        return {'total': float(total), 'message': (prefix + '\n\n' if prefix else '') + '\n'.join(receipt),
                'steps': s['steps'] + [f'price: goods={goods}; discount={discount}; delivery={fee}; total={total}']}

    def finish(s):
        return {'steps': s['steps'] + [f'finish: {s["status"]}']}

    builder = StateGraph(State, input_schema=OrderInput, output_schema=OrderOutput)
    for name, node in [('initialize', initialize), ('validate', validate), ('delivery', delivery),
                       ('check_stock', check_stock), ('substitute', substitute),
                       ('reject', reject), ('price', price), ('finish', finish)]:
        builder.add_node(name, node)
    builder.add_edge(START, 'initialize')
    builder.add_edge('initialize', 'validate')
    builder.add_conditional_edges('validate', lambda s: s['status'], {'accepted': 'delivery', 'rejected': 'finish'})
    builder.add_edge('delivery', 'check_stock')
    builder.add_conditional_edges('check_stock', route_stock, {'price': 'price', 'substitute': 'substitute', 'reject': 'reject'})
    builder.add_edge('substitute', 'check_stock')
    builder.add_edge('reject', 'finish')
    builder.add_edge('price', 'finish')
    builder.add_edge('finish', END)
    return builder.compile()


graph = build_graph()




In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print(f"PNG renderer unavailable: {exc}")
    print(graph.get_graph().draw_mermaid())

## Example orders and automated checks

In [ ]:
"""Run with: python -m unittest -v"""
import unittest


# label, item, quantity, day, city, status, final item, delivery, total, attempts
CASES = [
    ('delivery', 'challah', 2, 'Monday', 'Jerusalem', 'accepted', 'challah', 'delivery', 56, 0),
    ('pickup', 'sourdough', 2, 'Thursday', '', 'accepted', 'sourdough', 'pickup', 50, 0),
    ('discount', 'challah', 10, 'Monday', 'Jerusalem', 'accepted', 'challah', 'delivery', 173, 0),
    ('Friday', 'challah', 10, 'Friday', 'Jerusalem', 'accepted', 'challah', 'delivery', 200, 0),
    ('outside', 'challah', 2, 'Monday', 'Tel Aviv', 'accepted', 'challah', 'pickup', 36, 0),
    ('first substitute', 'babka', 2, 'Monday', '', 'accepted', 'rugelach', 'pickup', 60, 1),
    ('second substitute', 'babka', 4, 'Monday', '', 'accepted', 'sourdough', 'pickup', 100, 2),
    ('limit', 'babka', 5, 'Monday', '', 'rejected', 'sourdough', 'pickup', 0, 2),
    ('challah fallback', 'sourdough', 9, 'Monday', '', 'accepted', 'challah', 'pickup', 137.7, 1),
    ('no fallback', 'challah', 11, 'Monday', '', 'rejected', 'challah', 'pickup', 0, 0),
    ('discontinued', 'croissants', 1, 'Monday', '', 'rejected', 'croissants', 'pickup', 0, 0),
    ('unknown', 'pizza', 1, 'Monday', '', 'rejected', 'pizza', 'pickup', 0, 0),
    ('zero', 'challah', 0, 'Monday', '', 'rejected', 'challah', 'pickup', 0, 0),
    ('negative', 'challah', -2, 'Monday', '', 'rejected', 'challah', 'pickup', 0, 0),
]


class BakeryTests(unittest.TestCase):
    def test_orders(self):
        for label, item, quantity, day, city, status, final, mode, total, attempts in CASES:
            with self.subTest(label=label):
                result = graph.invoke(dict(item=item, quantity=quantity, day=day, city=city))
                self.assertEqual(set(result), {'status', 'item', 'delivery', 'total', 'message', 'steps'})
                self.assertEqual((result['status'], result['item'], result['delivery']), (status, final, mode))
                self.assertAlmostEqual(result['total'], total)
                self.assertEqual(sum(step.startswith('substitute:') for step in result['steps']), attempts)
                self.assertTrue(result['message'])
                self.assertEqual(result['steps'][-1], f'finish: {status}')
                if label == 'outside':
                    self.assertIn('only deliver inside Jerusalem', result['message'])

    def test_exact_discount_boundary(self):
        # Six sourdough loaves require a custom snapshot: initial stock is four.
        custom = build_graph({'challah': 10, 'babka': 0, 'rugelach': 3, 'sourdough': 7})
        for qty, expected in [(6, 170), (7, 168.75)]:
            result = custom.invoke(dict(item='sourdough', quantity=qty, day='Monday', city='Jerusalem'))
            self.assertEqual(result['total'], expected)

    def test_repeatable_and_malformed(self):
        order = dict(item=' CHALLAH ', quantity=10, day=' friday ', city=' jerusalem ')
        self.assertEqual(graph.invoke(order), graph.invoke(order))
        self.assertEqual(graph.invoke(order)['total'], 200)
        for qty in [True, 1.5, '2', None]:
            self.assertEqual(graph.invoke({**order, 'quantity': qty})['status'], 'rejected')

    def test_receipts(self):
        for label, item, quantity, day, city, status, final, mode, total, attempts in CASES:
            with self.subTest(label=label):
                message = graph.invoke(dict(item=item, quantity=quantity, day=day, city=city))['message']
                if status == 'rejected':
                    self.assertNotIn('Baked goods:', message)
                    self.assertNotIn('Total:', message)
                    continue
                self.assertIn(f'Total: {total:.2f}₪', message)
                self.assertIn('Jerusalem delivery: 20.00₪' if mode == 'delivery' else 'Pickup: 0.00₪', message)
                if attempts:
                    self.assertIn(f'Substitution: {item} → {final} (same quantity: {quantity}).', message)
                else:
                    self.assertNotIn('Substitution:', message)
                if label == 'discount':
                    self.assertIn('Baked goods: 10 × 18.00₪ = 180.00₪', message)
                    self.assertIn('Your 15% baked-goods discount: −27.00₪', message)
                elif label == 'Friday':
                    self.assertIn('Discount: 0.00₪ — discounts take a break on Fridays.', message)
                    self.assertNotIn('−27.00₪', message)
                elif label == 'pickup':
                    self.assertIn('baked goods exceed 150.00₪', message)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(BakeryTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()


## Your receipt page
Run the following cells to see every order as a designed receipt. Click **Download receipt page**, then open the downloaded HTML file for a full browser page. Edit the orders list to include your own orders.

In [ ]:
"""Build an offline HTML receipt gallery from actual graph results."""
from html import escape
from pathlib import Path


def render_receipts(orders):
    """orders contains (label, input dictionary, graph output dictionary) tuples."""
    cards = []
    accepted = sum(result['status'] == 'accepted' for _, _, result in orders)
    for number, (label, order, result) in enumerate(orders, 1):
        ok = result['status'] == 'accepted'
        status = 'Accepted' if ok else 'Cancelled'
        diary = ''.join(f'<li>{escape(step)}</li>' for step in result['steps'])
        message = ''.join(f'<div class="line{ " total-line" if line.startswith("Total:") else ""}">{escape(line) or "&nbsp;"}</div>'
                          for line in result['message'].splitlines())
        cards.append(f'''<article class="receipt">
          <div class="card-top"><span>ORDER {number:03d}</span><span class="badge {'ok' if ok else 'cancelled'}">{status}</span></div>
          <h2>{escape(str(label)).title()}</h2>
          <p class="meta">Requested: {escape(str(order['quantity']))} × {escape(str(order['item']))}<br>
          {escape(str(order['day']))} · {escape(str(order['city'])) or 'Pickup at the bakery'}</p>
          <div class="message">{message}</div>
          <div class="amount"><span>{'Amount to pay' if ok else 'Nothing to pay'}</span><strong>{result['total']:.2f}₪</strong></div>
          <details><summary>How we processed this order</summary><ol>{diary}</ol></details>
          </article>''')
    return '''<!doctype html><html lang="en"><head><meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>Dana's Bakery · Your receipts</title><style>
    *{box-sizing:border-box}body{margin:0;background:#f7f1e8;color:#35251e;font-family:Arial,sans-serif}
    header{background:#3e2923;color:#fff5e5;padding:42px max(24px,calc((100% - 1180px)/2))}
    .eyebrow{font-size:12px;text-transform:uppercase;letter-spacing:3px;color:#edc49a}
    h1{font-family:Georgia,serif;font-size:clamp(32px,5vw,54px);margin:12px 0}header p{color:#edd9c9;line-height:1.6}
    main{max-width:1228px;margin:auto;padding:28px 24px}.summary{display:flex;gap:14px;flex-wrap:wrap;margin-bottom:26px}
    .summary span{background:#ede1d2;padding:12px 18px;border-radius:12px}.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,340px),1fr));gap:24px;align-items:start}
    .receipt{background:#fffdf8;border:1px solid #e4d7c6;border-radius:18px;padding:26px;box-shadow:0 7px 22px #4630240a}
    .card-top{display:flex;justify-content:space-between;align-items:center;gap:10px;color:#806a5b;font-size:11px;letter-spacing:1px}
    .badge{padding:7px 10px;border-radius:20px;font-weight:bold;letter-spacing:0}.ok{background:#e3eee0;color:#335a34}.cancelled{background:#f7e2dc;color:#903f30}
    h2{font-family:Georgia,serif;font-size:27px;margin:22px 0 10px}.meta{color:#806a5b;font-size:13px;line-height:1.8;padding-bottom:18px;border-bottom:1px dashed #d6c4b0}
    .message{font-size:14px;line-height:1.7;overflow-wrap:anywhere}.total-line{font-weight:bold}.amount{display:flex;justify-content:space-between;align-items:center;gap:12px;margin-top:22px;padding:18px 0;border-top:1px dashed #d6c4b0}.amount span{color:#806a5b;font-size:13px}.amount strong{font-size:27px}
    details{border-top:1px solid #eee3d5;padding-top:16px;font-size:12px;color:#806a5b}summary{cursor:pointer}li{margin:9px 0;overflow-wrap:anywhere}
    footer{text-align:center;padding:32px;color:#806a5b;font-size:12px}
    @media print{body{background:white}header{background:white;color:#35251e}header p,.eyebrow{color:#806a5b}.summary{display:none}.grid{display:block}.receipt{break-inside:avoid;margin-bottom:20px;box-shadow:none}details{display:none}}
    </style></head><body><header><div class="eyebrow">Freshly baked · Made with care</div>
    <h1>Dana's Bakery</h1><p>Your orders, beautifully wrapped up.<br>Every total explained, every substitution shown.</p></header><main>
    <div class="summary">''' + f'<span>{len(orders)} orders</span><span>{accepted} accepted</span><span>{len(orders)-accepted} cancelled</span>' + '''</div>
    <section class="grid" aria-label="Order receipts">''' + ''.join(cards) + '''</section></main>
    <footer>With love, Dana · Independent quotes against the initial inventory · Print or save this page for your records.</footer></body></html>'''


def save_receipts(orders, filename='bakery_receipts.html'):
    path = Path(filename)
    path.write_text(render_receipts(orders), encoding='utf-8')
    return path




In [ ]:
from IPython.display import HTML, display
orders = []
for label, item, quantity, day, city, *_ in CASES:
    order = dict(item=item, quantity=quantity, day=day, city=city)
    orders.append((label, order, graph.invoke(order)))

# Add your own orders here before saving the page:
# order = dict(item='challah', quantity=10, day='Monday', city='Jerusalem')
# orders.append(('My order', order, graph.invoke(order)))

receipt_path = save_receipts(orders)
page = render_receipts(orders)
display(HTML('<iframe title="Dana bakery receipts" style="width:100%;height:1000px;border:0;border-radius:16px" srcdoc="' + escape(page, quote=True) + '"></iframe>'))

try:
    from google.colab import files
    import ipywidgets as widgets
    download = widgets.Button(description='Download receipt page', icon='download', layout=widgets.Layout(width='240px'))
    download.on_click(lambda _: files.download(str(receipt_path)))
    display(download)
    print('Click Download receipt page, then open bakery_receipts.html in your browser for a full page view.')
except ImportError:
    from IPython.display import FileLink
    display(FileLink(str(receipt_path)))
